# Move Nebula V3 to Kaggle

Import this notebook, then attach a **private Dataset** with these three existing files:

- `Nebula_V3_Data.pt`
- `v3-chat.pt` (latest: 33,012 saved steps)
- `v3-chat.best.pt` (best validation: 31,012 saved steps)

The first run verifies and restores the files. Training is **off by default**. No teacher keys, Internet downloads, Google Drive mount, or pretrained model are required.

Use **Save Version → Save & Run All** to execute a saved run and retain its completed output. Files live under `/kaggle/working/nebula/`. Saving checkpoints during an interactive session alone is not a durable backup. Keep downloaded copies too. GPU availability and session limits still apply.


In [ ]:
from pathlib import Path
import json,sys
import torch

RUN_TRAINING = False  # First migrate and save outputs. Enable only for a deliberate new experiment.
RUN_EVALUATION = False  # Optional: print test replies using the best checkpoint.
ADDITIONAL_STEPS = 1000
MAX_TRAINING_SECONDS = 1800
project = Path('/kaggle/working/nebula/api')
project.mkdir(parents=True,exist_ok=True)
torch.set_num_threads(2)
print({'GPU':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
       'project':str(project),'trainingEnabled':RUN_TRAINING})


In [ ]:
sources={'model/nebula.py': '"""Nebula\'s small, randomly initialized causal Transformer. No pretrained weights."""\nimport json\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nimport torch\nfrom torch import nn\n\nBOS, EOS, SYSTEM, USER, ASSISTANT, PAD = range(256, 262)\nVOCAB = 262\nROLES = {"system": SYSTEM, "user": USER, "assistant": ASSISTANT}\n\n@dataclass\nclass Config:\n    width: int = 128\n    heads: int = 4\n    layers: int = 2\n    context: int = 512\n\ndef config_for_preset(preset):\n    presets = {\n        \'smoke\': Config(width=64,heads=2,layers=1,context=256),\n        \'cpu\': Config(),\n        \'cpu-medium\': Config(width=192,heads=6,layers=4,context=1024),\n        \'cpu-large\': Config(width=256,heads=8,layers=6,context=1024),\n        \'gpu\': Config(width=384,heads=8,layers=6,context=2048),\n    }\n    if preset not in presets: raise ValueError(\'Unknown model preset\')\n    return presets[preset]\n\ndef encode(messages, answer_prefix=False):\n    tokens, learn = [BOS], [False]\n    for message in messages:\n        role = message["role"]\n        tokens.append(ROLES[role]); learn.append(False)\n        content = list(message["content"].encode("utf-8"))\n        tokens.extend(content); learn.extend([role == "assistant"] * len(content))\n        tokens.append(EOS); learn.append(role == "assistant")\n    if answer_prefix:\n        tokens.append(ASSISTANT); learn.append(False)\n    return tokens, learn\n\ndef pack_prompt(messages, context):\n    systems = [m for m in messages if m[\'role\'] == \'system\']\n    history = [m for m in messages if m[\'role\'] != \'system\']\n    if not history or history[-1][\'role\'] != \'user\': raise ValueError(\'A user question is required\')\n    prefix, _ = encode(systems)\n    selected = []\n    # Keep every system instruction and the full latest question; never silently cut rules.\n    available = context - len(prefix) - 2\n    for message in reversed(history):\n        chunk = encode([message])[0][1:]\n        if len(chunk) > available:\n            if not selected: raise ValueError(\'Rules and question exceed the model context\')\n            break\n        selected.insert(0, chunk); available -= len(chunk)\n    return prefix + [token for chunk in selected for token in chunk] + [ASSISTANT]\n\nclass Nebula(nn.Module):\n    def __init__(self, config):\n        super().__init__(); self.config = config\n        self.token = nn.Embedding(VOCAB, config.width)\n        self.position = nn.Embedding(config.context, config.width)\n        self.blocks = nn.ModuleList(nn.TransformerEncoderLayer(config.width, config.heads, config.width * 4, dropout=0.0, activation="gelu", batch_first=True, norm_first=True) for _ in range(config.layers))\n        self.norm = nn.LayerNorm(config.width)\n        self.output = nn.Linear(config.width, VOCAB, bias=False)\n        self.output.weight = self.token.weight\n        self.register_buffer(\'_causal_mask\',torch.ones(config.context,config.context,dtype=torch.bool).triu(1),persistent=False)\n        self.apply(self._initialize)\n\n    @staticmethod\n    def _initialize(module):\n        if isinstance(module, (nn.Linear, nn.Embedding)):\n            nn.init.normal_(module.weight, std=0.02)\n            if isinstance(module, nn.Linear) and module.bias is not None: nn.init.zeros_(module.bias)\n\n    def forward(self, tokens):\n        length = tokens.shape[1]\n        if length > self.config.context: raise ValueError("Context is too long")\n        value = self.token(tokens) + self.position(torch.arange(length, device=tokens.device))\n        causal = self._causal_mask[:length,:length]\n        for block in self.blocks: value = block(value, src_mask=causal)\n        return self.output(self.norm(value))\n\n    @torch.inference_mode()\n    def reply(self, messages, max_tokens=160, temperature=0.7):\n        self.eval()\n        encoded = pack_prompt(messages, self.config.context)\n        device = next(self.parameters()).device\n        tokens = torch.tensor([encoded], device=device)\n        result = []\n        for _ in range(min(max_tokens, self.config.context - len(encoded))):\n            logits = self(tokens)[:, -1, :].clone()\n            end = logits[:, EOS].clone(); logits[:, BOS:] = float(\'-inf\'); logits[:, EOS] = end\n            if temperature == 0: next_token = logits.argmax(-1, keepdim=True)\n            else: next_token = torch.multinomial(torch.softmax(logits / temperature, -1), 1)\n            value = int(next_token.item())\n            if value == EOS: break\n            result.append(value); tokens = torch.cat([tokens, next_token], dim=1)\n        return bytes(result).decode(\'utf-8\', errors=\'replace\').strip()\n\ndef save(model, path, metadata, training_state=None):\n    if any(not torch.isfinite(parameter).all() for parameter in model.parameters()):\n        raise ValueError(\'Checkpoint contains non-finite weights; existing saved weights were not replaced\')\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(\'.tmp\')\n    payload={\'format\': 1, \'config\': asdict(model.config), \'state\': model.state_dict(), \'metadata\': metadata}\n    if training_state is not None: payload[\'trainingState\']=training_state\n    torch.save(payload, temporary)\n    temporary.replace(path)\n\ndef load(path):\n    data = torch.load(path, map_location=\'cpu\', weights_only=True)\n    if data.get(\'format\') != 1: raise ValueError(\'Unsupported model checkpoint\')\n    model = Nebula(Config(**data[\'config\'])); model.load_state_dict(data[\'state\']); model.eval()\n    if any(not torch.isfinite(parameter).all() for parameter in model.parameters()):\n        raise ValueError(\'Checkpoint contains non-finite weights; resume from a healthy best checkpoint\')\n    return model, data.get(\'metadata\', {})\n', 'model/train.py': 'import argparse\nimport hashlib\nimport json\nimport random\nimport time\nimport itertools\nimport math\nfrom contextlib import nullcontext\nfrom pathlib import Path\nimport torch\nfrom torch.nn import functional as F\nfrom nebula import Config, Nebula, PAD, encode, load, save, config_for_preset\nfrom dataflow import read_jsonl, eligible\nfrom rules import with_owner_rules\n\nclass NumericalFailure(RuntimeError): pass\n\ndef examples(rows, context, include_owner_rules=False, objective=\'assistant\'):\n    result = []\n    for row in rows:\n        messages = row[\'messages\']\n        if objective == \'language\':\n            # Language pretraining uses continuous serialized text chunks, with no\n            # invented end-of-answer tokens at chunk boundaries. Chat tuning below\n            # uses complete replies instead.\n            prefix=encode(with_owner_rules([]) if include_owner_rules else [])[0]\n            body=encode(messages)[0][1:]\n            room=context+1-len(prefix)\n            if room<1: raise ValueError(\'Owner rules exceed model context\')\n            for offset in range(0,len(body),room):\n                tokens=prefix+body[offset:offset+room]\n                x=torch.tensor(tokens[:-1]); y=torch.tensor([-100]*(len(prefix)-1)+tokens[len(prefix):])\n                result.append((x,y))\n            continue\n        wrapped = with_owner_rules(messages) if include_owner_rules else messages\n        windows = [wrapped]\n        if len(encode(wrapped)[0]) > context + 1:\n            windows = []\n            for end in range(2,len(messages)+1,2):\n                for start in range(0,end,2):\n                    candidate=with_owner_rules(messages[start:end]) if include_owner_rules else messages[start:end]\n                    if len(encode(candidate)[0]) <= context+1:\n                        windows.append(candidate); break\n        for window in windows:\n            tokens, learn = encode(window)\n            x = torch.tensor(tokens[:-1]); y = torch.tensor([v if mask else -100 for v, mask in zip(tokens[1:], learn[1:])])\n            if (y != -100).any(): result.append((x, y))\n    return result\n\ndef partition_rows(rows, seed):\n    """Split original dialogues before creating windows, keeping duplicates together."""\n    training=[]; validation=[]\n    for row in rows:\n        identifier=hashlib.sha256(json.dumps(row[\'messages\'],sort_keys=True).encode()).hexdigest()\n        # Owned curriculum variants of one scenario stay in the same partition.\n        group=row.get(\'splitGroup\') if row.get(\'source\',{}).get(\'kind\')==\'seed\' else None\n        split_key=\'group:\'+group if isinstance(group,str) and group else identifier\n        score=int(hashlib.sha256((str(seed)+\':\'+split_key).encode()).hexdigest()[:8],16)\n        (validation if score%10==0 else training).append(row)\n    if not training or not validation: raise ValueError(\'Grouped split requires enough distinct conversations\')\n    return training,validation\n\ndef bucket_batches(data,batch_size,rng):\n    ordered=sorted(data,key=lambda pair:len(pair[0])); batches=[]\n    for offset in range(0,len(ordered),max(batch_size*32,32)):\n        bucket=ordered[offset:offset+max(batch_size*32,32)]; rng.shuffle(bucket)\n        batches.extend(bucket[i:i+batch_size] for i in range(0,len(bucket),batch_size))\n    rng.shuffle(batches)\n    return batches\n\ndef unique_windows(data,excluded=None):\n    seen=set(excluded or []); result=[]\n    for x,y in data:\n        identifier=hashlib.sha256(x.numpy().tobytes()+y.numpy().tobytes()).hexdigest()\n        if identifier in seen: continue\n        seen.add(identifier); result.append((x,y))\n    return result,seen\n\ndef loss(model, batch):\n    length = max(len(x) for x, _ in batch)\n    x = torch.full((len(batch), length), PAD, dtype=torch.long); y = torch.full_like(x, -100)\n    for i, (a, b) in enumerate(batch): x[i, :len(a)] = a; y[i, :len(b)] = b\n    device=next(model.parameters()).device\n    autocast=torch.autocast(\'cuda\',dtype=model._amp_dtype) if getattr(model,\'_amp_enabled\',False) else nullcontext()\n    with autocast:\n        return F.cross_entropy(model(x.to(device)).reshape(-1, 262), y.to(device).reshape(-1), ignore_index=-100)\n\ndef evaluate_loss(model, data, batch_size):\n    total = tokens = 0\n    for offset in range(0, len(data), batch_size):\n        batch = data[offset:offset+batch_size]\n        count = sum(int((y != -100).sum()) for _, y in batch)\n        total += float(loss(model, batch)) * count; tokens += count\n    return total / tokens if tokens else None\n\ndef train(args):\n    if args.steps < 1 or args.batch < 1 or args.threads < 1: raise ValueError(\'Positive steps, batch and threads required\')\n    if not math.isfinite(args.lr) or args.lr <= 0: raise ValueError(\'Learning rate must be finite and positive\')\n    continuous = getattr(args, \'continuous\', False)\n    save_every = getattr(args, \'save_every\', 100)\n    patience = getattr(args, \'patience\', 0)\n    if patience < 0: raise ValueError(\'Patience must be zero or positive\')\n    if save_every < 1: raise ValueError(\'Save interval must be positive\')\n    max_seconds=getattr(args,\'max_seconds\',0)\n    if not math.isfinite(max_seconds) or max_seconds<0: raise ValueError(\'Time limit must be zero or positive\')\n    torch.set_num_threads(args.threads); torch.manual_seed(args.seed); rng = random.Random(args.seed)\n    rows = read_jsonl(args.data)\n    snapshot = json.loads(Path(args.snapshot).read_text(encoding=\'utf-8\')) if args.snapshot else None\n    if any(row.get(\'source\', {}).get(\'kind\') == \'user\' for row in rows):\n        if continuous: raise ValueError(\'Continuous runs require seed/teacher data; user consent must be refreshed between bounded runs\')\n        if not snapshot or not 0 <= time.time() * 1000 - snapshot.get(\'createdAt\', 0) < 300000:\n            raise ValueError(\'User examples require a fresh consent check. Use scripts/model-data.mjs train.\')\n    # User records always require a fresh Firebase consent snapshot supplied by the wrapper.\n    rows = [row for row in rows if eligible(row, snapshot)]\n    if args.resume: model,resume_metadata=load(args.resume)\n    else: model,resume_metadata=Nebula(config_for_preset(args.preset)),{}\n    inherited_steps=resume_metadata.get(\'steps\',0)\n    requested_device=getattr(args,\'device\',\'cpu\')\n    device=\'cuda\' if requested_device==\'auto\' and torch.cuda.is_available() else \'cpu\' if requested_device==\'auto\' else requested_device\n    if device==\'cuda\' and not torch.cuda.is_available(): raise ValueError(\'CUDA GPU is unavailable in this PyTorch installation\')\n    model.to(device)\n    model._amp_enabled=device==\'cuda\' and getattr(args,\'amp\',False)\n    model._amp_dtype=torch.bfloat16 if device==\'cuda\' and torch.cuda.is_bf16_supported() else torch.float16\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=model._amp_enabled and model._amp_dtype==torch.float16)\n    objective=getattr(args,\'objective\',\'assistant\')\n    grouped=getattr(args,\'grouped_split\',False)\n    if grouped:\n        train_rows,validation_rows=partition_rows(rows,args.seed)\n        training=examples(train_rows,model.config.context,True,objective)\n        validation=examples(validation_rows,model.config.context,True,objective)\n        training,seen_windows=unique_windows(training)\n        validation,_=unique_windows(validation,seen_windows)\n        data=training+validation\n    else:\n        data = examples(rows, model.config.context, include_owner_rules=True,objective=objective)\n        rng.shuffle(data); split = max(1, int(len(data) * .9)); training, validation = data[:split], data[split:]\n    if not training: raise ValueError(\'No approved training examples fit the model context\')\n    if grouped and not validation: raise ValueError(\'No validation conversations fit the model context\')\n    available_validation=len(validation)\n    max_validation=getattr(args,\'max_validation\',0)\n    if max_validation:\n        validation_rng=random.Random(args.seed+1)\n        validation=validation_rng.sample(validation,min(len(validation),max_validation))\n    validation.sort(key=lambda pair:len(pair[0]))\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=.01)\n    with torch.no_grad(): initial = float(loss(model, training[:args.batch]))\n    if not math.isfinite(initial): initial=None\n    completed = 0\n    data_hash = hashlib.sha256(Path(args.data).read_bytes()).hexdigest()\n    best_path = Path(args.output).with_suffix(\'.best.pt\')\n    best_validation = float(\'inf\')\n    stale_checks = 0\n    batches=[]; epoch=0; completed_total_before=0\n    if best_path.exists():\n        _, previous = load(best_path)\n        if previous.get(\'dataSha256\') == data_hash and previous.get(\'seed\') == args.seed and previous.get(\'objective\',\'assistant\') == objective and previous.get(\'groupedSplit\',False) == grouped and previous.get(\'maxValidation\',0) == max_validation and previous.get(\'windowFormat\',1)==2:\n            previous_loss = previous.get(\'validationLoss\')\n            if isinstance(previous_loss, (int, float)): best_validation = previous_loss\n    state_enabled=getattr(args,\'save_training_state\',False)\n    if state_enabled and args.resume:\n        saved_state=torch.load(args.resume,map_location=\'cpu\',weights_only=True).get(\'trainingState\')\n        signature={\'dataSha256\':data_hash,\'objective\':objective,\'seed\':args.seed,\n                   \'batch\':args.batch,\'bucketed\':getattr(args,\'bucketed\',False),\n                   \'groupedSplit\':grouped,\'windowFormat\':2,\'maxValidation\':max_validation,\n                   \'amp\':model._amp_enabled,\'learningRate\':args.lr}\n        if saved_state and saved_state.get(\'signature\')==signature:\n            optimizer.load_state_dict(saved_state[\'optimizer\'])\n            scaler.load_state_dict(saved_state[\'scaler\'])\n            rng.setstate(saved_state[\'samplerRandom\'])\n            torch.set_rng_state(saved_state[\'torchRandom\'])\n            if device==\'cuda\' and saved_state.get(\'cudaRandom\'):\n                torch.cuda.set_rng_state_all(saved_state[\'cudaRandom\'])\n            batches=[[training[index] for index in batch] for batch in saved_state[\'remainingBatches\']]\n            epoch=saved_state[\'epoch\']; stale_checks=saved_state[\'staleChecks\']\n            completed_total_before=saved_state.get(\'stageSteps\',0)\n            print(json.dumps({\'trainingStateRestored\':True,\'previousStageSteps\':completed_total_before}),flush=True)\n        else:\n            print(json.dumps({\'trainingStateRestored\':False,\'reason\':\'legacy_checkpoint_or_new_training_configuration\'}),flush=True)\n    signature={\'dataSha256\':data_hash,\'objective\':objective,\'seed\':args.seed,\n               \'batch\':args.batch,\'bucketed\':getattr(args,\'bucketed\',False),\n               \'groupedSplit\':grouped,\'windowFormat\':2,\'maxValidation\':max_validation,\n               \'amp\':model._amp_enabled,\'learningRate\':args.lr}\n    training_indices={id(x):index for index,(x,_) in enumerate(training)}\n    def checkpoint(stopped=False,validate=True,reason=None):\n        nonlocal best_validation, stale_checks\n        model.eval()\n        current=validation_loss=None\n        if validate:\n            with torch.no_grad():\n                current = float(loss(model, training[:args.batch]))\n                validation_loss = evaluate_loss(model, validation, args.batch)\n            if not math.isfinite(current) or validation_loss is not None and not math.isfinite(validation_loss):\n                raise NumericalFailure(\'Non-finite checkpoint metric\')\n        report = {\'parameters\': sum(p.numel() for p in model.parameters()), \'steps\': inherited_steps+completed,\'stepsThisRun\':completed, \'trainingExamples\': len(training), \'validationExamples\': len(validation),\'availableValidationExamples\':available_validation,\'originalConversations\':len(rows), \'skippedExamples\': None if grouped else max(0,len(rows) - len(data)), \'initialLoss\': initial, \'finalLoss\': current, \'validationLoss\': validation_loss, \'seed\': args.seed, \'dataSha256\': data_hash, \'experimental\': True, \'qualityApproved\': False, \'stopped\': stopped,\'objective\':objective,\'groupedSplit\':grouped,\'maxValidation\':max_validation,\'device\':device,\'windowFormat\':2}\n        if reason: report[\'stopReason\']=reason\n        improved=validation_loss is not None and validation_loss < best_validation\n        if improved:\n            best_validation = validation_loss\n            stale_checks = 0\n        elif validation_loss is not None:\n            stale_checks += 1\n            if validation_loss > best_validation * 1.2:\n                print(\'Validation loss is worsening: this run is overfitting. More varied data is needed; Ctrl+C saves and stops.\', flush=True)\n        training_state=None\n        if state_enabled:\n            training_state={\'signature\':signature,\'optimizer\':optimizer.state_dict(),\'scaler\':scaler.state_dict(),\n                \'samplerRandom\':rng.getstate(),\'torchRandom\':torch.get_rng_state(),\n                \'cudaRandom\':torch.cuda.get_rng_state_all() if device==\'cuda\' else [],\n                \'remainingBatches\':[[training_indices[id(x)] for x,_ in batch] for batch in batches],\n                \'epoch\':epoch,\'staleChecks\':stale_checks,\'stageSteps\':completed_total_before+completed}\n            report[\'stageSteps\']=completed_total_before+completed\n            report[\'trainingStateIncluded\']=True\n        if state_enabled: save(model,args.output,report,training_state=training_state)\n        else: save(model,args.output,report)\n        if improved:\n            if state_enabled: save(model,best_path,report,training_state=training_state)\n            else: save(model,best_path,report)\n            print(json.dumps({\'bestCheckpoint\': str(best_path), \'validationLoss\': best_validation}), flush=True)\n        print(json.dumps({\'checkpoint\': str(args.output), **report}), flush=True)\n    # Save even before the first update, then periodically and on Ctrl+C.\n    started=time.monotonic(); trained_tokens=0\n    try:\n        # Resume already has a verified checkpoint and optimizer state. Avoid an\n        # extra validation check consuming patience before its first update.\n        if not (state_enabled and completed_total_before): checkpoint()\n        for step in itertools.count() if continuous else range(args.steps):\n            if getattr(args,\'bucketed\',False):\n                if not batches: batches=bucket_batches(training,args.batch,rng); epoch+=1\n                batch=batches.pop()\n            else: batch = [training[rng.randrange(len(training))] for _ in range(args.batch)]\n            model.train()\n            optimizer.zero_grad(set_to_none=True); value = loss(model, batch)\n            if not torch.isfinite(value): raise NumericalFailure(\'Non-finite training loss\')\n            scaler.scale(value).backward(); scaler.unscale_(optimizer)\n            gradient_norm=torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            if not torch.isfinite(gradient_norm): raise NumericalFailure(\'Non-finite gradients; optimizer update was blocked\')\n            scaler.step(optimizer); scaler.update()\n            completed = step + 1\n            trained_tokens+=sum(len(x) for x,_ in batch)\n            if step == 0 or completed % 25 == 0: print(json.dumps({\'step\': completed, \'loss\': round(float(value.detach()), 4),\'epoch\':epoch or None,\'tokensPerSecond\':round(trained_tokens/max(.001,time.monotonic()-started))}), flush=True)\n            if max_seconds and time.monotonic()-started>=max_seconds:\n                checkpoint(stopped=True,validate=False,reason=\'time_limit\')\n                return {\'steps\':completed,\'stopped\':True,\'reason\':\'time_limit\'}\n            if completed % save_every == 0:\n                checkpoint()\n                if patience and stale_checks >= patience:\n                    print(json.dumps({\'stopped\':\'validation_not_improving\',\'steps\':completed,\'bestCheckpoint\':str(best_path),\'note\':\'Latest weights and best validation weights are saved. Live website unchanged.\'}),flush=True)\n                    return {\'steps\':completed,\'stopped\':True,\'reason\':\'validation_not_improving\'}\n    except KeyboardInterrupt:\n        checkpoint(stopped=True,validate=False,reason=\'interrupted\')\n        print(\'Stopped. Latest weights saved; the live website was not changed.\', flush=True)\n        return {\'steps\': completed, \'stopped\': True}\n    except NumericalFailure:\n        optimizer.zero_grad(set_to_none=True)\n        try:\n            checkpoint(stopped=True,validate=False,reason=\'non_finite_loss_or_gradient\')\n            message=\'Stopped for a numerical error. Finite current weights saved without validation; the best checkpoint is preserved.\'\n        except ValueError:\n            message=\'Stopped for a numerical error. Invalid weights were not saved; existing checkpoints are preserved.\'\n        print(message+\' Resume from a healthy best checkpoint with a lower learning rate.\',flush=True)\n        return {\'steps\':completed,\'stopped\':True,\'reason\':\'non_finite_loss_or_gradient\'}\n    checkpoint()\n    report=load(args.output)[1]\n    return report\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=\'Train owned Nebula weights from scratch. CPU preset is experimental, not production quality.\')\n    parser.add_argument(\'--continuous\', action=\'store_true\'); parser.add_argument(\'--save-every\', type=int, default=100)\n    parser.add_argument(\'--save-training-state\',action=\'store_true\',help=\'Save and restore compatible optimizer, sampler and random states\')\n    parser.add_argument(\'--max-seconds\',type=float,default=0,help=\'Stop and save after this many seconds; 0 disables\')\n    parser.add_argument(\'--patience\', type=int, default=0, help=\'Stop after this many validation checks without improvement; 0 disables\')\n    parser.add_argument(\'--device\',choices=[\'auto\',\'cpu\',\'cuda\'],default=\'auto\')\n    parser.add_argument(\'--objective\',choices=[\'assistant\',\'language\'],default=\'assistant\')\n    parser.add_argument(\'--grouped-split\',action=\'store_true\'); parser.add_argument(\'--bucketed\',action=\'store_true\'); parser.add_argument(\'--max-validation\',type=int,default=0)\n    parser.add_argument(\'--data\', required=True); parser.add_argument(\'--output\', required=True); parser.add_argument(\'--snapshot\'); parser.add_argument(\'--resume\'); parser.add_argument(\'--preset\', choices=[\'cpu\', \'smoke\',\'cpu-medium\',\'cpu-large\',\'gpu\'], default=\'cpu\'); parser.add_argument(\'--steps\', type=int, default=500); parser.add_argument(\'--batch\', type=int, default=4); parser.add_argument(\'--threads\', type=int, default=2); parser.add_argument(\'--seed\', type=int, default=42); parser.add_argument(\'--lr\', type=float, default=.0005)\n    train(parser.parse_args())\n', 'model/dataflow.py': '"""Reviewed datasets. These commands never call teachers with user chat content."""\nimport hashlib\nimport json\nimport re\nfrom pathlib import Path\nfrom urllib.request import Request, urlopen\nfrom urllib.parse import urlparse\nimport os\nimport time\nfrom teacher_http import request_json\nfrom rules import owner_instructions\n\ndef read_jsonl(path):\n    with Path(path).open(encoding=\'utf-8\') as stream:\n        return [json.loads(line) for line in stream if line.strip()]\n\ndef write_jsonl(path, rows):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(\'.tmp\')\n    temporary.write_text(\'\'.join(json.dumps(row, ensure_ascii=False) + \'\\n\' for row in rows), encoding=\'utf-8\')\n    temporary.replace(path)\n\ndef redact(text):\n    text = re.sub(r\'\\b[\\w.+-]+@[\\w.-]+\\.[a-zA-Z]{2,}\\b\', \'[email removed]\', text)\n    text = re.sub(r\'\\b(?:sk-|AIza)[\\w-]{15,}\\b\', \'[credential removed]\', text)\n    text = re.sub(r\'\\bBearer\\s+[\\w.~/+=-]{12,}\', \'[credential removed]\', text, flags=re.I)\n    text = re.sub(r\'(?<!\\w)(?:\\+?\\d[\\d ()-]{7,}\\d)(?!\\w)\', \'[number removed]\', text)\n    return text\n\ndef example_id(messages):\n    return hashlib.sha256(json.dumps(messages, sort_keys=True).encode()).hexdigest()\n\ndef eligible(row, snapshot=None):\n    if row.get(\'reviewStatus\') != \'approved\': return False\n    source = row.get(\'source\', {})\n    if source.get(\'kind\') == \'user\':\n        if not snapshot: return False\n        profile = snapshot.get(\'accounts\', {}).get(source.get(\'accountKey\'))\n        owner = snapshot.get(\'training\', {}).get(source.get(\'owner\'))\n        if not profile or profile.get(\'improve\') is not True or not owner or owner.get(\'revoked\'): return False\n        if profile.get(\'trainingGeneration\', 0) != source.get(\'generation\'): return False\n        original = owner.get(\'samples\', {}).get(source.get(\'chatId\'), {}).get(source.get(\'turnId\'))\n        if not original or original.get(\'consentVersion\') != 1: return False\n        # An export cannot be modified into different content while claiming old consent.\n        messages = [{\'role\': m[\'role\'], \'content\': redact(m[\'content\'])} for m in original.get(\'messages\', [])]\n        # Reviewers may correct the answer, but cannot replace the consented user prompt.\n        if not messages or not row.get(\'messages\') or row[\'messages\'][0] != messages[0]: return False\n    elif source.get(\'kind\') == \'teacher\':\n        if source.get(\'trainingAllowed\') is not True: return False\n    elif source.get(\'kind\') == \'public\':\n        allowed = (source.get(\'dataset\') == \'HuggingFaceTB/everyday-conversations-llama3.1-2k\' and source.get(\'subset\') == \'default\') or (source.get(\'dataset\') == \'HuggingFaceTB/smoltalk\' and source.get(\'subset\') == \'smol-magpie-ultra\')\n        story = source.get(\'dataset\') == \'roneneldan/TinyStories\' and source.get(\'subset\') == \'default\' and source.get(\'license\') == \'cdla-sharing-1.0\'\n        if not ((allowed and source.get(\'license\') == \'apache-2.0\') or story) or not source.get(\'recordId\'): return False\n    elif source.get(\'kind\') != \'seed\': return False\n    messages = row.get(\'messages\')\n    # User exports remain a single consent-checked pair. Owned/teacher datasets may be dialogues.\n    maximum = 2 if source.get(\'kind\') == \'user\' else 12\n    return (isinstance(messages, list) and 2 <= len(messages) <= maximum and len(messages) % 2 == 0\n            and all(isinstance(m, dict) and m.get(\'role\') == (\'user\' if i % 2 == 0 else \'assistant\')\n                    and isinstance(m.get(\'content\'), str) and 0 < len(m[\'content\'].strip()) <= 16000\n                    for i, m in enumerate(messages)))\n\ndef prepare(inputs, output, snapshot=None):\n    rows, seen = [], set()\n    for path in inputs:\n        for row in read_jsonl(path):\n            if not eligible(row, snapshot): continue\n            identifier = example_id(row[\'messages\'])\n            if identifier in seen: continue\n            seen.add(identifier); rows.append({**row, \'id\': identifier})\n    if not rows: raise ValueError(\'No approved, currently eligible training examples\')\n    write_jsonl(output, rows)\n    return len(rows)\n\ndef generate_teacher(config, prompts, output, limit=10, transport=None):\n    """Bounded calls to an OpenAI-compatible API. Original prompts only."""\n    if not 1 <= limit <= 100: raise ValueError(\'Request limit must be 1 to 100\')\n    if config.get(\'trainingAllowed\') is not True: raise ValueError(\'Choose a teacher whose output may be used for training\')\n    endpoint = config[\'url\']; parsed = urlparse(endpoint)\n    if parsed.username or parsed.password or parsed.query: raise ValueError(\'Credentials belong in an environment variable\')\n    if parsed.scheme != \'https\' and not (parsed.scheme == \'http\' and parsed.hostname in [\'localhost\', \'127.0.0.1\', \'::1\']): raise ValueError(\'HTTPS required for remote teacher\')\n    key = os.environ.get(config.get(\'keyEnv\', \'NEBULA_TEACHER_KEY\'), \'\')\n    if not key and parsed.hostname not in [\'localhost\', \'127.0.0.1\', \'::1\']: raise ValueError(\'Teacher API key is missing\')\n    rows = read_jsonl(output) if Path(output).exists() else []\n    completed = {r.get(\'promptId\') for r in rows}\n    calls = 0\n    for prompt in prompts:\n        # Prevent accidental transmission of exported user conversations to teachers.\n        if prompt.get(\'source\') != \'original\' or not isinstance(prompt.get(\'prompt\'), str): raise ValueError(\'Teacher prompts must be original curriculum prompts\')\n        text = prompt[\'prompt\'].strip()\n        if not text or len(text) > 4000: raise ValueError(\'Invalid curriculum prompt\')\n        identifier = hashlib.sha256((config.get(\'name\', \'custom\') + \':\' + config[\'model\'] + \':\' + text).encode()).hexdigest()\n        if identifier in completed: continue\n        if calls >= limit: break\n        payload = {\'model\': config[\'model\'], \'messages\': [{\'role\': \'system\', \'content\': \'Create a training example for Nebula using these owner instructions.\\n\' + owner_instructions() + \'\\n\' + config.get(\'instructions\', \'\')}, {\'role\': \'user\', \'content\': text}], \'max_tokens\': min(int(config.get(\'maxTokens\', 256)), 1024), \'temperature\': 0.6}\n        if config.get(\'temperature\', 0.6) is None: payload.pop(\'temperature\')\n        if config.get(\'reasoningEffort\') in [\'low\',\'medium\',\'high\']: payload[\'reasoning_effort\'] = config[\'reasoningEffort\']\n        request = Request(endpoint, json.dumps(payload).encode(), {\'Content-Type\': \'application/json\', \'User-Agent\':\'Nebula-Teacher/1.0\', **({\'Authorization\': \'Bearer \' + key} if key else {})})\n        if calls and not transport:\n            delay = min(60, max(0, float(config.get(\'requestIntervalSeconds\', 0))))\n            if delay: time.sleep(delay)\n        if transport: data = transport(request)\n        else:\n            # Never follow a redirect carrying a credential to another host.\n            from urllib.request import build_opener, HTTPRedirectHandler\n            class NoRedirect(HTTPRedirectHandler):\n                def redirect_request(self, *args, **kwargs): return None\n            data = request_json(build_opener(NoRedirect), request, provider=config.get(\'name\',\'teacher\'), max_retries=int(config.get(\'httpRetries\',2)))\n        answer = data[\'choices\'][0][\'message\'][\'content\']\n        if not isinstance(answer, str) or not answer.strip() or len(answer) > 16000: raise ValueError(\'Invalid teacher answer\')\n        rows.append({\'promptId\': identifier, \'reviewStatus\': \'pending\', \'messages\': [{\'role\': \'user\', \'content\': text}, {\'role\': \'assistant\', \'content\': redact(answer)}], \'source\': {\'kind\': \'teacher\', \'provider\': config.get(\'name\', \'custom\'), \'model\': config[\'model\'], \'trainingAllowed\': True}})\n        calls += 1; completed.add(identifier); write_jsonl(output, rows)\n    return calls\n', 'model/teacher_http.py': '"""Bounded HTTP retries and credential-free quota diagnostics."""\nimport json\nimport math\nimport re\nimport time\nfrom urllib.error import HTTPError\n\nclass TeacherRequestError(ValueError):\n    def __init__(self, message, status=429, retry_after=60, retryable=True):\n        super().__init__(message)\n        self.status = status\n        self.retry_after = retry_after\n        self.retryable = retryable\n\ndef quota_info(error):\n    result = {\'zero\': False, \'daily\': False, \'retry\': None}\n    try:\n        raw = error.read(200001)\n        if len(raw) > 200000: return result\n        value = json.loads(raw).get(\'error\', {})\n        message = str(value.get(\'message\', \'\'))\n        result[\'zero\'] = bool(re.search(r\'limit\\s*:\\s*0(?:\\D|$)\', message, re.I))\n        for detail in value.get(\'details\', []):\n            if not isinstance(detail, dict): continue\n            if str(detail.get(\'@type\', \'\')).endswith(\'RetryInfo\'):\n                match = re.fullmatch(r\'(\\d+(?:\\.\\d+)?)s\', str(detail.get(\'retryDelay\', \'\')))\n                if match: result[\'retry\'] = math.ceil(float(match[1]))\n            if str(detail.get(\'@type\', \'\')).endswith(\'QuotaFailure\'):\n                for violation in detail.get(\'violations\', []):\n                    if not isinstance(violation, dict): continue\n                    if str(violation.get(\'quotaValue\', \'\')) == \'0\': result[\'zero\'] = True\n                    if \'perday\' in str(violation.get(\'quotaId\', \'\')).lower(): result[\'daily\'] = True\n        header = error.headers.get(\'Retry-After\') if error.headers else None\n        if header and header.isdigit(): result[\'retry\'] = max(result[\'retry\'] or 0, int(header))\n    except (ValueError, TypeError, AttributeError, OSError): pass\n    return result\n\ndef request_json(opener, request, sleep=time.sleep, provider=\'gemini\', max_retries=2):\n    label = \'Groq\' if provider == \'groq\' else \'Gemini secondary\' if provider == \'gemini-secondary\' else \'Gemini\' if provider == \'gemini\' else \'Teacher\'\n    quota_url = \'https://console.groq.com/settings/limits\' if provider == \'groq\' else \'https://aistudio.google.com/rate-limit\' if provider in [\'gemini\',\'gemini-secondary\'] else \'your provider quota dashboard\'\n    for attempt in range(max_retries + 1):\n        try:\n            with opener.open(request, timeout=60) as response:\n                raw = response.read(200001)\n                if len(raw) > 200000: raise ValueError(\'Teacher reply too large\')\n                return json.loads(raw)\n        except HTTPError as error:\n            if error.code != 429: raise\n            info = quota_info(error)\n            if info[\'zero\']:\n                raise TeacherRequestError(f\'{label} HTTP 429: this model has a zero quota limit for your project. Check {quota_url} and select a model with available quota. Retrying cannot change a zero limit.\', retryable=False) from None\n            if info[\'daily\']:\n                raise TeacherRequestError(f\'{label} HTTP 429: daily quota exhausted. Check {quota_url} for the reset. Completed examples remain saved.\', retryable=False) from None\n            delay = max(15 * (2 ** attempt), info[\'retry\'] or 0)\n            if attempt == max_retries or delay > 60:\n                hint = f\' Provider suggests waiting {info["retry"]} seconds.\' if info[\'retry\'] is not None else \'\'\n                raise TeacherRequestError(f\'{label} HTTP 429: rate or quota limit still blocks requests.\' + hint + f\' Check {quota_url}. Completed examples remain saved.\', retry_after=max(60,info[\'retry\'] or 0)) from None\n            print(f\'{label} rate limited. Waiting {delay} seconds before retry {attempt + 1}/2. Ctrl+C stops.\', flush=True)\n            sleep(delay)\n', 'model/rules.py': '"""Owner instructions shared by training and private inference."""\nimport json\nimport os\nfrom pathlib import Path\n\ndef owner_instructions():\n    value = json.loads((Path(__file__).resolve().parent.parent / \'nebula-rules.json\').read_text(encoding=\'utf-8\'))\n    lines = value.get(\'instructions\')\n    if not isinstance(lines, list) or not lines or any(not isinstance(line, str) or not line.strip() for line in lines): raise ValueError(\'Invalid owner rules\')\n    extra = os.environ.get(\'NEBULA_OWNER_INSTRUCTIONS\', \'\')\n    text = \'\\n\'.join(lines + ([extra] if extra else []))\n    if len(text) > 16000: raise ValueError(\'Owner instructions too long\')\n    return text\n\ndef with_owner_rules(messages):\n    text = owner_instructions()\n    if messages and messages[0][\'role\'] == \'system\' and (messages[0][\'content\'] == text or messages[0][\'content\'].startswith(text + \'\\n\')): return messages\n    return [{\'role\': \'system\', \'content\': text}, *messages]\n', 'model/evaluate.py': '"""Inspect a local checkpoint without teacher calls or deployment."""\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\n\nruntime = Path(__file__).resolve().parents[4] / \'work/nebula-python-deps\'\nif runtime.joinpath(\'torch\').is_dir(): sys.path.insert(0, str(runtime))\nimport torch\nfrom nebula import load\nfrom rules import with_owner_rules\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--checkpoint\', default=str(Path(__file__).resolve().parent/\'checkpoints/nebula-continuous.pt\'))\n    parser.add_argument(\'--device\', choices=[\'auto\',\'cpu\',\'cuda\'], default=\'auto\')\n    parser.add_argument(\'--temperature\', type=float, default=.7)\n    args = parser.parse_args()\n    torch.set_num_threads(2)\n    model, metadata = load(args.checkpoint)\n    device=\'cuda\' if args.device==\'auto\' and torch.cuda.is_available() else \'cpu\' if args.device==\'auto\' else args.device\n    model.to(device)\n    print(json.dumps({\'checkpoint\': args.checkpoint, \'trainingSteps\': metadata.get(\'steps\'), \'validationLoss\': metadata.get(\'validationLoss\'), \'experimental\': True}), flush=True)\n    for prompt in [\'hi\', \'What is your name?\', \'What is 2 + 2?\', \'What is 7 + 8?\', \'Explain gravity simply.\', \'Can you help me write an email?\', \'My name is Sam. What is my name?\', \'Why is the sky blue?\', \'Tell me a short story.\', \'How do I boil an egg?\']:\n        torch.manual_seed(42)\n        reply = model.reply(with_owner_rules([{\'role\': \'user\', \'content\': prompt}]), max_tokens=120, temperature=args.temperature)\n        print(json.dumps({\'prompt\': prompt, \'reply\': reply}, ensure_ascii=True), flush=True)\n\nif __name__ == \'__main__\': main()\n', 'model/kaggle_migrate.py': '"""Restore verified V3 data and checkpoints into Kaggle\'s saved output folder."""\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport shutil\nimport zlib\n\nimport torch\nfrom nebula import load\n\nDATA_FILES={\n    \'model/data/stories-approved.jsonl\',\'model/data/improved-chat.jsonl\',\n    \'model/data/v3-language.jsonl\',\'model/data/v3-chat.jsonl\',\n    \'model/data/v3-foundations.jsonl\',\'model/data/v3-conversations.jsonl\',\n    \'model/data/stories-sources.json\',\'model/data/public-sources.json\',\n    \'model/DATA_SOURCES.md\',\'nebula-rules.json\',\n}\n\ndef sha256(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\ndef restore_data(backup,project):\n    payload=torch.load(backup,map_location=\'cpu\',weights_only=True)\n    if payload.get(\'format\')!=\'nebula-v3-data-1\' or payload.get(\'compression\')!=\'zlib\':\n        raise ValueError(\'This is not a Nebula V3 data backup\')\n    entries=payload.get(\'files\',{}); hashes=payload.get(\'sha256\',{})\n    if set(entries)!=set(hashes) or not set(entries)<=DATA_FILES:\n        raise ValueError(\'Unexpected files in data backup\')\n    required={\'model/data/v3-chat.jsonl\',\'model/data/v3-language.jsonl\',\'nebula-rules.json\'}\n    if not required<=set(entries): raise ValueError(\'Required training data or rules are missing\')\n    decoded={}; total=0\n    for name,tensor in entries.items():\n        if not isinstance(tensor,torch.Tensor) or tensor.dtype!=torch.uint8 or tensor.ndim!=1:\n            raise ValueError(\'Invalid compressed data entry\')\n        decoder=zlib.decompressobj()\n        value=decoder.decompress(tensor.numpy().tobytes(),64*1024*1024+1)\n        total+=len(value)\n        if not decoder.eof or decoder.unused_data or total>256*1024*1024:\n            raise ValueError(\'Oversized or incomplete data backup\')\n        if hashlib.sha256(value).hexdigest()!=hashes[name]: raise ValueError(\'Training data checksum mismatch\')\n        decoded[name]=value\n    project=Path(project).resolve()\n    # Validate all existing targets before writing any file.\n    for name,value in decoded.items():\n        target=project/name\n        if not target.resolve().is_relative_to(project): raise ValueError(\'Unsafe destination\')\n        if target.exists() and sha256(target)!=hashes[name]:\n            raise ValueError(\'Existing training data differs; choose a new project folder\')\n    for name,value in decoded.items():\n        target=project/name; target.parent.mkdir(parents=True,exist_ok=True)\n        if target.exists(): continue\n        temporary=target.with_suffix(target.suffix+\'.restore-tmp\')\n        temporary.write_bytes(value); temporary.replace(target)\n    return hashes\n\ndef checkpoint_info(path,data_hash):\n    model,metadata=load(path); del model\n    if metadata.get(\'dataSha256\')!=data_hash or metadata.get(\'objective\')!=\'assistant\':\n        raise ValueError(\'Checkpoint does not match the backed-up V3 chat dataset\')\n    return metadata\n\ndef find_backup(input_root):\n    paths=sorted(Path(input_root).rglob(\'Nebula_V3_Data.pt\'))\n    if not paths: raise FileNotFoundError(\'Add your private dataset containing Nebula_V3_Data.pt and both v3-chat checkpoint files.\')\n    if len({sha256(p) for p in paths})!=1:\n        raise ValueError(\'Different V3 data backups are attached. Keep only the intended dataset version.\')\n    return paths[0]\n\ndef atomic_copy(source,target):\n    source=Path(source); target=Path(target)\n    if source.resolve()==target.resolve(): return\n    target.parent.mkdir(parents=True,exist_ok=True)\n    temporary=target.with_suffix(target.suffix+\'.copy-tmp\')\n    shutil.copyfile(source,temporary); temporary.replace(target)\n\ndef migrate(input_root,project):\n    input_root=Path(input_root); project=Path(project)\n    backup=find_backup(input_root)\n    hashes=restore_data(backup,project)\n    data_hash=hashes[\'model/data/v3-chat.jsonl\']\n    checkpoints=project/\'model/checkpoints\'\n    selected={}; summary={}\n    for filename in [\'v3-chat.pt\',\'v3-chat.best.pt\']:\n        candidates=list(input_root.rglob(filename))\n        target=checkpoints/filename\n        if target.exists(): candidates.append(target)\n        compatible=[]\n        for path in candidates:\n            info=checkpoint_info(path,data_hash)\n            compatible.append((path,info))\n        if not compatible: raise FileNotFoundError(\'Missing \'+filename+\'. Add both V3 model files as notebook inputs.\')\n        if filename.endswith(\'.best.pt\'):\n            def score(item):\n                value=item[1].get(\'validationLoss\')\n                return value if isinstance(value,(float,int)) and math.isfinite(value) else float(\'inf\')\n            source,info=min(compatible,key=score)\n        else: source,info=max(compatible,key=lambda item:item[1].get(\'steps\',0))\n        selected[target]=source\n        summary[filename]={\'savedSteps\':info.get(\'steps\'),\'validationLoss\':info.get(\'validationLoss\'),\n                           \'trainingStateIncluded\':info.get(\'trainingStateIncluded\',False)}\n    # Only install model files after both inputs have passed validation.\n    for target,source in selected.items(): atomic_copy(source,target)\n    atomic_copy(backup,project/\'Nebula_V3_Data.pt\')\n    summary[\'dataSha256\']=data_hash\n    summary[\'project\']=str(project)\n    summary[\'trainingStarted\']=False\n    (project/\'migration-report.json\').write_text(json.dumps(summary,indent=2),encoding=\'utf-8\')\n    print(json.dumps(summary,indent=2),flush=True)\n    return summary\n'}
for name,content in sources.items():
    target=project/name; target.parent.mkdir(parents=True,exist_ok=True)
    target.write_text(content,encoding='utf-8')
sys.path.insert(0,str(project/'model'))
for module in ['nebula','train','dataflow','teacher_http','rules','evaluate','kaggle_migrate']:
    sys.modules.pop(module,None)
from kaggle_migrate import migrate
report=migrate(Path('/kaggle/input'),project)


## Optional bounded training

Leave `RUN_TRAINING=False` for the migration run. The latest checkpoint contains optimizer and sampler state. The saved V3 run had already stopped improving, so more repetitions are not a guaranteed quality upgrade. This optional run preserves the existing best model unless validation improves, saves every 200 updates, and stops after 1,000 additional updates, 30 minutes, or stalled validation. Only one GPU is used; selecting T4 ×2 does not automatically double speed.

To continue a future saved run, attach this notebook's saved output as an additional input. The restore cell chooses the latest matching checkpoint and the best matching validation checkpoint. Also retain the original private Dataset as input.


In [ ]:
if RUN_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError('Enable a GPU accelerator in Kaggle settings before training.')
    from types import SimpleNamespace
    from train import train
    checkpoint=project/'model/checkpoints/v3-chat.pt'
    result=train(SimpleNamespace(
        data=project/'model/data/v3-chat.jsonl',output=checkpoint,resume=checkpoint,
        snapshot=None,preset='cpu-large',steps=ADDITIONAL_STEPS,batch=16,threads=2,
        seed=42,lr=.00005,continuous=False,save_every=200,patience=10,
        device='cuda',amp=True,objective='assistant',grouped_split=True,bucketed=True,
        max_validation=256,save_training_state=True,max_seconds=MAX_TRAINING_SECONDS))
    print(result)
else:
    print('Migration only: existing weights and optimizer state have been preserved.')


In [ ]:
if RUN_EVALUATION:
    import subprocess
    result=subprocess.run([sys.executable,str(project/'model/evaluate.py'),
        '--checkpoint',str(project/'model/checkpoints/v3-chat.best.pt')],
        capture_output=True,text=True,check=True)
    print(result.stdout)
    print(result.stderr)


## Confirm and retain the output

After **Save & Run All** completes, open that version's **Output** and confirm the following files exist. You can download them or attach this saved notebook output to another notebook. Do not assume unsaved interactive files survive a reset. The original uploaded Dataset is separate from the temporary working directory.


In [ ]:
for filename in ['model/checkpoints/v3-chat.pt','model/checkpoints/v3-chat.best.pt',
                 'Nebula_V3_Data.pt','migration-report.json']:
    path=project/filename
    if not path.is_file(): raise FileNotFoundError(path)
    print(str(path),f'{path.stat().st_size/1_000_000:.1f} MB')
print('Finished. Confirm these files in the saved notebook version Output.')
